# Rice Disease Detection — Training Experiments
**Anna University Final Year Project 2024–25**

This notebook is the **clean reference** for training both models:
- **Part 1** — EfficientNetB0 CNN (disease classification, 4 classes)
- **Part 2** — WeatherRiskLSTM (weather-driven binary risk, 5 Tamil Nadu districts)
- **Part 3** — Fusion Ablation (grid-search alpha on validation set)

All hyperparameters are defined in `config.yaml` at the project root.
Run cells top-to-bottom in Google Colab with a GPU runtime.

---
**Key design decisions (documented in source notebooks):**
- CNN uses `CategoricalFocalCrossentropy` (γ=2, label_smoothing=0.1) — not CrossEntropy
- CNN prediction = `argmax(softmax)` only — **no confidence threshold**
- LSTM output = raw `sigmoid(logit)` — **no hard threshold**
- Fusion alpha validated via grid search, not set manually


---
## Part 1 — EfficientNetB0 CNN Training


In [ ]:
# ── Cell 1.1 — Install & Imports ────────────────────────────────────────────
import os, shutil, random, warnings
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")


In [ ]:
# ── Cell 1.2 — Configuration (mirrors config.yaml) ──────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
NUM_CLASSES    = 4
PHASE1_EPOCHS  = 30
PHASE2_EPOCHS  = 20
LR_PHASE1      = 1e-4   # Lower than 1e-3 — avoids ±8% val oscillation with focal loss
LR_PHASE2      = 1e-5
UNFREEZE_TOP_N = 20     # Top N layers of EfficientNetB0 unfrozen in Phase 2

DRIVE_PATH    = "/content/drive/MyDrive/Updated_rice_disease/"
RAW_DATA_PATH = "/content/data/raw"
TRAIN_DIR     = os.path.join(RAW_DATA_PATH, "train")
VAL_DIR       = os.path.join(RAW_DATA_PATH, "val")
TEST_DIR      = os.path.join(RAW_DATA_PATH, "test")
MODEL_SAVE_PATH = os.path.join(DRIVE_PATH, "rice_v3_efficientnet.keras")
BEST_MODEL_PATH = os.path.join(DRIVE_PATH, "rice_v3_efficientnet_best.keras")
TFLITE_PATH     = os.path.join(DRIVE_PATH, "rice_v3.tflite")

# CRITICAL: class names MUST be alphabetically ordered.
# TF loads classes alphabetically from directory names.
# Changing this order corrupts all predictions silently.
CLASS_NAMES = [
    "Bacterial Leaf Blight",   # index 0  alpha=0.15
    "Brown Spot",              # index 1  alpha=0.20
    "Healthy Rice Leaf",       # index 2  alpha=0.10  ← HEALTHY_IDX
    "Leaf Blast",              # index 3  alpha=0.35  ← hardest class
]
HEALTHY_IDX    = 2
LEAF_BLAST_IDX = 3

FOCAL_GAMMA          = 2.0
FOCAL_LABEL_SMOOTHING = 0.1
FOCAL_ALPHA = [0.15, 0.20, 0.10, 0.35]  # order matches CLASS_NAMES above

os.makedirs(DRIVE_PATH, exist_ok=True)
print("Config loaded.")
print(f"  Focal alpha : {dict(zip(CLASS_NAMES, FOCAL_ALPHA))}")


In [ ]:
# ── Cell 1.3 — Mount Drive + Download Dataset ────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists("/content/Rice_Leaf_AUG"):
    if os.path.exists("/content/kaggle.json"):
        import subprocess
        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        shutil.copy("/content/kaggle.json", os.path.join(kaggle_dir, "kaggle.json"))
        os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
        subprocess.run(["kaggle", "datasets", "download", "anshulm257/rice-disease-dataset", "-q"])
        subprocess.run(["unzip", "-q", "rice-disease-dataset.zip", "-d", "/content/"])
        print("Dataset downloaded.")
    else:
        print("Place dataset manually at /content/Rice_Leaf_AUG/")
else:
    print("Dataset already present.")


In [ ]:
# ── Cell 1.4 — Dataset Split (70 / 15 / 15) ─────────────────────────────────
SOURCE_DIR = "/content/Rice_Leaf_AUG"
CLEAN_DIR  = "/content/clean_rice"
TARGET_CLASSES = ["Bacterial Leaf Blight", "Leaf Blast", "Brown Spot", "Healthy Rice Leaf"]

def copy_selected_classes(source_dir, dest_dir, class_list):
    os.makedirs(dest_dir, exist_ok=True)
    for cls in class_list:
        src = os.path.join(source_dir, cls)
        dst = os.path.join(dest_dir, cls)
        if not os.path.isdir(src):
            print(f"  [SKIP] '{cls}' not found."); continue
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = len([f for f in os.listdir(dst) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  [OK] {cls:<30} — {n} images")

def split_dataset(source_dir, output_dir, train_ratio=0.70, val_ratio=0.15, seed=SEED):
    random.seed(seed)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split), exist_ok=True)
    print(f"{'Class':<30} {'Train':>6} {'Val':>5} {'Test':>5} {'Total':>6}")
    print("-" * 55)
    for cls in sorted(os.listdir(source_dir)):
        cls_path = os.path.join(source_dir, cls)
        if not os.path.isdir(cls_path): continue
        imgs = sorted(os.listdir(cls_path)); random.shuffle(imgs)
        n = len(imgs); t_end = int(n*train_ratio); v_end = t_end + int(n*val_ratio)
        splits = {"train": imgs[:t_end], "val": imgs[t_end:v_end], "test": imgs[v_end:]}
        for split_name, file_list in splits.items():
            dst = os.path.join(output_dir, split_name, cls)
            os.makedirs(dst, exist_ok=True)
            for f in file_list:
                shutil.copy2(os.path.join(cls_path, f), os.path.join(dst, f))
        c = {s: len(v) for s, v in splits.items()}
        print(f"{cls:<30} {c['train']:>6} {c['val']:>5} {c['test']:>5} {n:>6}")
    print("\nSplit complete.")

copy_selected_classes(SOURCE_DIR, CLEAN_DIR, TARGET_CLASSES)
split_dataset(CLEAN_DIR, RAW_DATA_PATH)


In [ ]:
# ── Cell 1.5 — Data Loading (label_mode=categorical for Focal Loss) ───────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="categorical",  # one-hot required by CategoricalFocalCrossentropy
    shuffle=True, seed=SEED
)
val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=False
)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=False
)

class_names = train_ds.class_names

# Safety asserts — guard against wrong class order forever
assert class_names == sorted(class_names), f"Class order wrong: {class_names}"
assert class_names.index("Healthy Rice Leaf") == HEALTHY_IDX
assert class_names.index("Leaf Blast")        == LEAF_BLAST_IDX
print("✓ Class order verified:", class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

label_counts = Counter()
for _, labels in train_ds.unbatch():
    label_counts[int(np.argmax(labels.numpy()))] += 1
total_train = sum(label_counts.values())
print(f"\nTraining class distribution:")
for idx, cls in enumerate(class_names):
    cnt = label_counts.get(idx, 0)
    print(f"  {cls:<30} {cnt:>5}  ({100*cnt/total_train:.1f}%)")


In [ ]:
# ── Cell 1.6 — Augmentation ─────────────────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, seed=SEED),       # ±54° — valid for field images
    layers.RandomZoom(0.10, seed=SEED),
    layers.RandomTranslation(0.05, 0.05, seed=SEED),
    layers.RandomContrast(0.15, seed=SEED),
    layers.RandomBrightness(0.15, seed=SEED),
], name="data_augmentation")
print("Augmentation layer defined.")


In [ ]:
# ── Cell 1.7 — Build Model ──────────────────────────────────────────────────
def build_model(lr=LR_PHASE1):
    """
    EfficientNetB0 feature extractor + classification head.
    Built-in preprocessing handles 0–255 float input — no manual rescaling.
    """
    base_model = EfficientNetB0(
        input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False  # Phase 1: freeze backbone

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image_input")
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout_1")(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                     name="dense_256")(x)
    x = layers.Dropout(0.2, name="dropout_2")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(x)

    model = keras.Model(inputs, outputs, name="RiceLeaf_v3_EfficientNetB0")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.CategoricalFocalCrossentropy(
            gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA,
            label_smoothing=FOCAL_LABEL_SMOOTHING, from_logits=False
        ),
        metrics=["accuracy"]
    )
    return model

model = build_model()
model.summary(line_length=90)
total = model.count_params()
trainable = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f"\nTotal params    : {total:,}")
print(f"Trainable (P1)  : {trainable:,}  (head only)")
print(f"Frozen          : {total - trainable:,}")


In [ ]:
# ── Cell 1.8 — Phase 1 Training (Frozen Backbone) ───────────────────────────
callbacks_p1 = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True,
                  mode="min", verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                      min_lr=1e-7, mode="min", verbose=1),
    ModelCheckpoint(filepath=BEST_MODEL_PATH, monitor="val_accuracy",
                    save_best_only=True, mode="max", verbose=1),
]

print("=" * 60)
print(f"PHASE 1 : Head only | LR={LR_PHASE1} | Epochs up to {PHASE1_EPOCHS}")
print("=" * 60)

history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS, callbacks=callbacks_p1, verbose=1
)


In [ ]:
# ── Cell 1.9 — Phase 2 Fine-tuning (Unfreeze Top 20 Layers) ────────────────
base_model = model.get_layer("efficientnetb0")
base_model.trainable = True

# Unfreeze top UNFREEZE_TOP_N layers; keep BatchNorm frozen
freeze_until = len(base_model.layers) - UNFREEZE_TOP_N
for i, layer in enumerate(base_model.layers):
    layer.trainable = (i >= freeze_until)
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False  # critical for small datasets

n_trainable = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f"Fine-tuning {n_trainable:,} parameters")
print(f"Unfreezing layers [{freeze_until} : {len(base_model.layers)-1}]")

BEST_FT_PATH = BEST_MODEL_PATH.replace(".keras", "_finetuned.keras")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE2),
    loss=keras.losses.CategoricalFocalCrossentropy(
        gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA,
        label_smoothing=FOCAL_LABEL_SMOOTHING, from_logits=False
    ),
    metrics=["accuracy"]
)
callbacks_p2 = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True,
                  mode="min", verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=4,
                      min_lr=1e-8, mode="min", verbose=1),
    ModelCheckpoint(filepath=BEST_FT_PATH, monitor="val_accuracy",
                    save_best_only=True, mode="max", verbose=1),
]
p1_actual = len(history_p1.history["accuracy"])
print(f"\nPhase 1 ran for {p1_actual} epochs.")
print("=" * 60)
print(f"PHASE 2 : Fine-tune | LR={LR_PHASE2} | +{PHASE2_EPOCHS} epochs")
print("=" * 60)

history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=p1_actual + PHASE2_EPOCHS, initial_epoch=p1_actual,
    callbacks=callbacks_p2, verbose=1
)


In [ ]:
# ── Cell 1.10 — Training Curves ─────────────────────────────────────────────
def combine_histories(h1, h2):
    return {k: h1.history[k] + h2.history[k] for k in h1.history}

history = combine_histories(history_p1, history_p2)
p1_len  = len(history_p1.history["accuracy"])
epochs  = range(1, len(history["accuracy"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("EfficientNetB0 v3 — Focal Loss Training History", fontsize=13, fontweight="bold")
for ax, (train_k, val_k), title in zip(
    axes, [("accuracy","val_accuracy"),("loss","val_loss")], ["Accuracy","Loss"]
):
    ax.plot(epochs, history[train_k], label="Train",      color="#2196F3", lw=2)
    ax.plot(epochs, history[val_k],   label="Validation", color="#F44336", lw=2)
    ax.axvline(p1_len, color="gray", ls="--", lw=1.5, label=f"Fine-tune start (ep {p1_len})")
    ax.axvspan(0,      p1_len,      alpha=0.05, color="blue")
    ax.axvspan(p1_len, len(epochs), alpha=0.05, color="green")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, "training_history.png"), dpi=150)
plt.show()


In [ ]:
# ── Cell 1.11 — Evaluation ──────────────────────────────────────────────────
# Load best checkpoint (not necessarily the final epoch)
best_path = BEST_FT_PATH if os.path.exists(BEST_FT_PATH) else BEST_MODEL_PATH
print(f"Loading best model: {best_path}")
model_eval = keras.models.load_model(best_path)

y_true_oh    = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_true       = np.argmax(y_true_oh, axis=1)
y_pred_probs = model_eval.predict(test_ds, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)  # argmax only — no threshold

acc       = np.mean(y_true == y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall    = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1        = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\n" + "=" * 55)
print("       TEST SET EVALUATION — EfficientNetB0 v3")
print("=" * 55)
print(f"  Accuracy           : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Weighted Precision : {precision:.4f}")
print(f"  Weighted Recall    : {recall:.4f}")
print(f"  Weighted F1        : {f1:.4f}")
print("=" * 55)
print("\nPer-Class Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


In [ ]:
# ── Cell 1.12 — Confusion Matrix ─────────────────────────────────────────────
conf_mat      = confusion_matrix(y_true, y_pred)
conf_mat_norm = conf_mat.astype("float") / conf_mat.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix — EfficientNetB0 Test Set", fontsize=14, fontweight="bold")
kw = dict(annot=True, xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
sns.heatmap(conf_mat,      fmt="d",   cmap="Blues",  ax=axes[0], **kw)
axes[0].set_title("Raw Counts"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_xticklabels(class_names, rotation=30, ha="right")
sns.heatmap(conf_mat_norm, fmt=".2f", cmap="Greens", ax=axes[1], **kw, vmin=0, vmax=1)
axes[1].set_title("Normalised (diagonal = per-class Recall)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_xticklabels(class_names, rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, "confusion_matrix.png"), dpi=150)
plt.show()


In [ ]:
# ── Cell 1.13 — Grad-CAM ─────────────────────────────────────────────────────
import matplotlib.cm as cm

LAST_CONV_LAYER = "top_conv"

def build_gradcam_model(model):
    """Dual-output model: one forward pass → correct gradient tape."""
    base_model   = model.get_layer("efficientnetb0")
    conv_layer   = base_model.get_layer(LAST_CONV_LAYER)
    backbone_dual = tf.keras.Model(
        inputs=base_model.input,
        outputs=[conv_layer.output, base_model.output], name="backbone_dual"
    )
    img_input = model.input
    aug_out   = model.get_layer("data_augmentation")(img_input, training=False)
    conv_maps_out, bb_out = backbone_dual(aug_out, training=False)
    x = model.get_layer("gap")(bb_out)
    x = model.get_layer("dropout_1")(x,    training=False)
    x = model.get_layer("dense_256")(x)
    x = model.get_layer("dropout_2")(x,    training=False)
    preds_out = model.get_layer("classifier")(x)
    return tf.keras.Model(inputs=img_input, outputs=[conv_maps_out, preds_out])

def compute_gradcam(gradcam_model, img_array, class_idx):
    img_tensor = tf.cast(img_array, tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        conv_maps, preds = gradcam_model(img_tensor, training=False)
        tape.watch(conv_maps)
        class_score = preds[0, class_idx]
    grads   = tape.gradient(class_score, conv_maps)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.nn.relu(conv_maps[0] @ weights[..., tf.newaxis])
    heatmap = tf.squeeze(heatmap).numpy()
    if heatmap.max() > 0: heatmap /= heatmap.max()
    return heatmap

def make_overlay(raw_img_array, heatmap, alpha=0.45):
    hm_pil  = Image.fromarray(np.uint8(255 * heatmap), mode="L")
    hm_big  = np.array(hm_pil.resize((224, 224), Image.BILINEAR)) / 255.0
    colored = np.uint8(255 * plt.cm.get_cmap("jet")(hm_big)[:, :, :3])
    return np.uint8(raw_img_array[0] * (1 - alpha) + colored * alpha)

# Build GradCAM model and visualise one image per class
gradcam_model = build_gradcam_model(model_eval)
fig, axes = plt.subplots(4, 3, figsize=(13, 17))
fig.suptitle("Grad-CAM — EfficientNetB0 v3\nRed = model attention area",
             fontsize=13, fontweight="bold", y=1.01)

for row, cls in enumerate(class_names):
    cls_dir = os.path.join(TEST_DIR, cls)
    files = sorted([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    img   = Image.open(os.path.join(cls_dir, files[0])).convert("RGB").resize((224, 224))
    arr   = np.array(img, dtype=np.float32)
    batch = np.expand_dims(arr, 0)
    cidx  = class_names.index(cls)
    _, preds = gradcam_model(tf.cast(batch, tf.float32), training=False)
    pred_idx  = int(np.argmax(preds[0]))
    conf      = float(preds[0][pred_idx]) * 100
    heatmap   = compute_gradcam(gradcam_model, batch, pred_idx)
    overlay   = make_overlay(batch, heatmap)
    correct   = "✅" if pred_idx == cidx else "❌"
    axes[row][0].imshow(arr.astype(np.uint8)); axes[row][0].set_ylabel(cls[:20], fontsize=9)
    axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    axes[row][1].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    axes[row][1].set_xlabel(f"max={heatmap.max():.3f}", fontsize=8)
    axes[row][1].set_xticks([]); axes[row][1].set_yticks([])
    axes[row][2].imshow(overlay)
    axes[row][2].set_xlabel(f"{correct} {class_names[pred_idx][:14]} ({conf:.0f}%)", fontsize=8)
    axes[row][2].set_xticks([]); axes[row][2].set_yticks([])

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, "gradcam.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Cell 1.14 — Save Model + TFLite Export ──────────────────────────────────
best_path = BEST_FT_PATH if os.path.exists(BEST_FT_PATH) else BEST_MODEL_PATH
model = keras.models.load_model(best_path)
model.save(MODEL_SAVE_PATH)
print(f"Keras model saved : {MODEL_SAVE_PATH}  ({os.path.getsize(MODEL_SAVE_PATH)/1e6:.1f} MB)")

# TFLite export (for mobile / edge deployment)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"TFLite saved : {TFLITE_PATH}  ({len(tflite_model)/1024:.0f} KB)")

# Copy to trained_models/ for the project (update path as needed)
# shutil.copy(MODEL_SAVE_PATH, "trained_models/rice_cnn_model.keras")
# shutil.copy(TFLITE_PATH,     "trained_models/rice_cnn_model.tflite")


---
## Part 2 — WeatherRiskLSTM Training


In [ ]:
# ── Cell 2.1 — LSTM Imports ─────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, f1_score
import pickle
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")


In [ ]:
# ── Cell 2.2 — Fetch Weather Data (Open-Meteo Archive API) ──────────────────
# Install if needed: !pip install openmeteo-requests requests-cache retry-requests

import openmeteo_requests, requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo     = openmeteo_requests.Client(session=retry_session)

DISTRICTS = [
    {"name": "Thanjavur",    "lat": 10.79, "lon": 79.14, "district_id": 1},
    {"name": "Nilgiris",     "lat": 11.49, "lon": 76.73, "district_id": 2},
    {"name": "Chennai",      "lat": 13.08, "lon": 80.27, "district_id": 3},
    {"name": "Virudhunagar", "lat":  9.59, "lon": 77.96, "district_id": 4},
    {"name": "Nagapattinam", "lat": 10.77, "lon": 79.84, "district_id": 5},
]

WEATHER_VARS = [
    "weather_code", "temperature_2m_max", "temperature_2m_min",
    "precipitation_sum", "rain_sum", "precipitation_hours",
    "et0_fao_evapotranspiration", "sunshine_duration",
    "wind_speed_10m_max", "wind_gusts_10m_max",
    "relative_humidity_2m_mean", "pressure_msl_mean",
    "soil_moisture_0_to_7cm_mean"
]

all_dfs = []
for dist in DISTRICTS:
    print(f"Fetching: {dist['name']}...")
    params = {
        "latitude": dist["lat"], "longitude": dist["lon"],
        "start_date": "2015-01-01", "end_date": "2024-12-31",
        "daily": WEATHER_VARS, "timezone": "auto",
    }
    responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
    response  = responses[0]; daily = response.Daily()
    daily_data = {"date": pd.date_range(
        start=pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()), inclusive="left"
    )}
    for idx, var in enumerate(WEATHER_VARS):
        daily_data[var] = daily.Variables(idx).ValuesAsNumpy()
    df_dist = pd.DataFrame(daily_data)
    df_dist["district_id"]   = dist["district_id"]
    df_dist["district_name"] = dist["name"]
    all_dfs.append(df_dist)
    print(f"  → {len(df_dist)} rows")

daily_dataframe = pd.concat(all_dfs, ignore_index=True)
daily_dataframe.to_csv("daily_data.csv", index=False)
print(f"\n✓ Saved daily_data.csv — {len(daily_dataframe)} rows")


In [ ]:
# ── Cell 2.3 — Preprocessing Pipeline ───────────────────────────────────────
import pandas as pd

df = pd.read_csv("daily_data.csv")
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert("Asia/Kolkata").dt.normalize()
df["date"] = df["date"].dt.tz_localize(None)
df = df[(df["date"] >= "2015-01-01") & (df["date"] <= "2024-12-31")]
df = df.sort_values(["district_id", "date"]).reset_index(drop=True)

# Step 3 — Disease risk labels (IRRI / Kim et al. 2018 / EPIRICE-SB 2020)
df["risk_brown_spot"] = (
    (df["relative_humidity_2m_mean"] > 86) &
    (df["temperature_2m_max"].between(16, 36)) &
    ((df["precipitation_sum"] < 5) | (df["precipitation_hours"] > 8))
).astype(int)

df["risk_leaf_blast"] = (
    (df["relative_humidity_2m_mean"] > 90) &
    (df["temperature_2m_max"].between(17, 28)) &
    (df["precipitation_hours"] > 10) &
    (df["sunshine_duration"] < 21600)
).astype(int)

df["risk_bacterial_blight"] = (
    (df["relative_humidity_2m_mean"] > 70) &
    (df["temperature_2m_max"].between(25, 34)) &
    (df["precipitation_sum"] > 20) &
    (df["wind_gusts_10m_max"] > 35)
).astype(int)

df["risk_label"] = (
    (df["risk_brown_spot"]==1) | (df["risk_leaf_blast"]==1) | (df["risk_bacterial_blight"]==1)
).astype(int)

combined = df["risk_label"].sum()
print(f"Risk days: {combined} / {len(df)} ({combined/len(df)*100:.1f}%)")
print(f"pos_weight = {(len(df)-combined)/combined:.2f}")

# Steps 4–6 — Feature engineering
df["month"]       = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["day_sin"]   = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["day_cos"]   = np.cos(2 * np.pi * df["day_of_year"] / 365)
df.drop(columns=["month","day_of_year"], inplace=True)

def get_season(month):
    if month in [6,7,8,9]:  return 0
    if month in [10,11,12]: return 1
    if month in [1,2]:      return 2
    return 3
df["season"]          = df["date"].dt.month.apply(get_season)
df["district_id_norm"] = df["district_id"] / 5.0   # H3 FIX: district identity
df["precip_7d_sum"]   = df["precipitation_sum"].rolling(window=7, min_periods=1).sum()

FEATURE_COLS = [
    "weather_code","temperature_2m_max","temperature_2m_min","precipitation_sum",
    "rain_sum","precipitation_hours","et0_fao_evapotranspiration","sunshine_duration",
    "wind_speed_10m_max","wind_gusts_10m_max","relative_humidity_2m_mean",
    "pressure_msl_mean","soil_moisture_0_to_7cm_mean",
    "month_sin","month_cos","day_sin","day_cos","season","precip_7d_sum",
    "district_id_norm"  # input_size = 20
]
assert len(FEATURE_COLS) == 20, f"Expected 20 features, got {len(FEATURE_COLS)}"

scaler = MinMaxScaler()
df[FEATURE_COLS] = scaler.fit_transform(df[FEATURE_COLS])
with open("scaler.pkl", "wb") as f: pickle.dump(scaler, f)
print(f"Scaled {len(FEATURE_COLS)} features → [0,1].  Saved scaler.pkl")

# Step 8 — Sequences per district (H3 FIX: no cross-district leakage)
LOOKBACK = 14
X, y = [], []
for dist_id in sorted(df["district_id"].unique()):
    df_d = df[df["district_id"] == dist_id].reset_index(drop=True)
    for i in range(LOOKBACK, len(df_d)):
        X.append(df_d[FEATURE_COLS].iloc[i-LOOKBACK:i].values)
        y.append(df_d["risk_label"].iloc[i])

X = np.array(X, dtype=np.float32); y = np.array(y, dtype=np.float32)
print(f"X shape : {X.shape}   y shape : {y.shape}")

# Step 9 — Split 70/15/15
n = len(X); train_end = int(n*0.70); val_end = int(n*0.85)
X_train, y_train = X[:train_end],        y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],          y[val_end:]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

for name, arr in [("X_train",X_train),("y_train",y_train),
                  ("X_val",X_val),("y_val",y_val),("X_test",X_test),("y_test",y_test)]:
    np.save(f"{name}.npy", arr)
print("\nAll arrays saved.")


In [ ]:
# ── Cell 2.4 — LSTM Model Definition ────────────────────────────────────────
class WeatherRiskLSTM(nn.Module):
    """
    input_size=20 (H3 FIX — includes district_id_norm).
    Do NOT change architecture without retraining.
    """
    def __init__(self, input_size=20, hidden_size=64, num_layers=2, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1)
        )
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.classifier(lstm_out[:, -1, :]).squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_lstm = WeatherRiskLSTM().to(device)
print(f"LSTM params : {sum(p.numel() for p in model_lstm.parameters() if p.requires_grad):,}")
print(f"Device      : {device}")


In [ ]:
# ── Cell 2.5 — LSTM Training ─────────────────────────────────────────────────
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

# WeightedRandomSampler — oversample risk days
class_counts  = np.bincount(y_train.astype(int))
sample_weights = torch.tensor((1.0/class_counts)[y_train.astype(int)], dtype=torch.float32)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, sampler=sampler)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=32, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=32, shuffle=False)

no_risk_count = int((y_train==0).sum()); risk_count = int(y_train.sum())
pos_weight = torch.tensor([no_risk_count/risk_count], dtype=torch.float32)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer  = torch.optim.Adam(model_lstm.parameters(), lr=0.001, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

EPOCHS=60; EARLY_STOP=15
# THRESHOLD used here ONLY for training-time F1 tracking.
# Inference returns raw sigmoid — no threshold applied there.
THRESHOLD=0.40

best_val_loss=float('inf'); patience_counter=0
train_losses=[]; val_losses=[]; train_f1s=[]; val_f1s=[]

for epoch in range(1, EPOCHS+1):
    model_lstm.train(); t_loss, t_preds, t_labels = 0,[],[]
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad(); out=model_lstm(Xb); loss=criterion(out,yb)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model_lstm.parameters(),1.0); optimizer.step()
        t_loss += loss.item()*Xb.size(0)
        pred = (torch.sigmoid(out)>=THRESHOLD).float()
        t_preds.extend(pred.cpu().numpy()); t_labels.extend(yb.cpu().numpy())

    model_lstm.eval(); v_loss, v_preds, v_labels = 0,[],[]
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out=model_lstm(Xb); loss=criterion(out,yb)
            v_loss += loss.item()*Xb.size(0)
            pred = (torch.sigmoid(out)>=THRESHOLD).float()
            v_preds.extend(pred.cpu().numpy()); v_labels.extend(yb.cpu().numpy())

    avg_t=t_loss/len(X_train); avg_v=v_loss/len(X_val)
    t_f1=f1_score(t_labels,t_preds,zero_division=0); v_f1=f1_score(v_labels,v_preds,zero_division=0)
    train_losses.append(avg_t); val_losses.append(avg_v)
    train_f1s.append(t_f1);    val_f1s.append(v_f1)
    scheduler.step(avg_v)

    if avg_v < best_val_loss:
        best_val_loss=avg_v; patience_counter=0
        torch.save(model_lstm.state_dict(), "best_lstm_v2.pth")
    else:
        patience_counter += 1

    if epoch%5==0 or epoch==1:
        print(f"  Ep {epoch:3d}/{EPOCHS} | TL:{avg_t:.4f} TF1:{t_f1:.3f} | VL:{avg_v:.4f} VF1:{v_f1:.3f}")

    if patience_counter >= EARLY_STOP:
        print(f"\n  Early stopping at epoch {epoch}"); break

print(f"\nBest val loss: {best_val_loss:.4f}  Saved: best_lstm_v2.pth")
# shutil.copy("best_lstm_v2.pth", "trained_models/rice_lstm_model.pth")
# shutil.copy("scaler.pkl",       "trained_models/scaler.pkl")


In [ ]:
# ── Cell 2.6 — LSTM Test Evaluation ─────────────────────────────────────────
model_lstm.load_state_dict(torch.load("best_lstm_v2.pth", map_location=device))
model_lstm.eval()

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        probs = torch.sigmoid(model_lstm(Xb))
        # Note: threshold only for evaluation reporting — NOT used at inference time
        preds = (probs >= THRESHOLD).float()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["No Risk","Risk"], digits=3))
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds); tn,fp,fn,tp = cm.ravel()
print(f"TN={tn}  FP={fp}\nFN={fn}  TP={tp}")


---
## Part 3 — Fusion Ablation Study

Grid-search alpha ∈ {0.0, 0.1, …, 1.0} on the **validation set only**.
Select the best alpha, then apply it **once** to the test set.
Update `config.yaml → fusion.alpha` with the result.


In [ ]:
# ── Cell 3.1 — Load Both Models for Ablation ────────────────────────────────
# Load CNN
best_cnn_path = BEST_FT_PATH if os.path.exists(BEST_FT_PATH) else BEST_MODEL_PATH
cnn_model = keras.models.load_model(best_cnn_path)

# Load LSTM
lstm_model_ab = WeatherRiskLSTM(input_size=20).to(device)
lstm_model_ab.load_state_dict(torch.load("best_lstm_v2.pth", map_location=device))
lstm_model_ab.eval()

# Load image val/test sets
val_ds_ab = keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=False
)
test_ds_ab = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=False
)

y_val_true  = np.argmax(np.concatenate([y.numpy() for _,y in val_ds_ab],  axis=0), axis=1)
y_test_true = np.argmax(np.concatenate([y.numpy() for _,y in test_ds_ab], axis=0), axis=1)

# CNN risk scores (1 - P_healthy)
cnn_val_probs  = cnn_model.predict(val_ds_ab,  verbose=0)
cnn_test_probs = cnn_model.predict(test_ds_ab, verbose=0)
cnn_val_risk   = 1.0 - cnn_val_probs[:,  HEALTHY_IDX]
cnn_test_risk  = 1.0 - cnn_test_probs[:, HEALTHY_IDX]

# LSTM sequences
X_val_seq  = np.load("X_val.npy");  y_val_seq  = np.load("y_val.npy")
X_test_seq = np.load("X_test.npy"); y_test_seq = np.load("y_test.npy")

def lstm_predict_proba(model, X_np, device, batch_size=256):
    model.eval(); scores = []
    with torch.no_grad():
        for start in range(0, len(X_np), batch_size):
            batch  = torch.tensor(X_np[start:start+batch_size], dtype=torch.float32).to(device)
            probs  = torch.sigmoid(model(batch)).cpu().numpy()
            scores.extend(probs.tolist())
    return np.array(scores, dtype=np.float32)

lstm_val_scores  = lstm_predict_proba(lstm_model_ab, X_val_seq,  device)
lstm_test_scores = lstm_predict_proba(lstm_model_ab, X_test_seq, device)
print("Models loaded. Starting ablation sweep...")


In [ ]:
# ── Cell 3.2 — Alpha Grid Search on Validation Set ──────────────────────────
# WARNING: We use VAL set for alpha selection. Test set is untouched until 3.3.
# Using test set here would be data leakage.

ALPHA_VALUES = [round(a * 0.1, 1) for a in range(11)]  # 0.0, 0.1, ..., 1.0

# Align sample lengths (CNN and LSTM val sets may differ)
n_val = min(len(cnn_val_risk), len(lstm_val_scores))
cnn_r   = cnn_val_risk[:n_val]
lstm_r  = lstm_val_scores[:n_val]
# For binary comparison: any non-healthy CNN prediction = risk
y_val_binary = (y_val_true[:n_val] != HEALTHY_IDX).astype(int)

results = []
print(f"{'Alpha':>6}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
print("-" * 38)
for alpha in ALPHA_VALUES:
    fused = alpha * cnn_r + (1.0 - alpha) * lstm_r
    preds = (fused >= 0.5).astype(int)
    from sklearn.metrics import precision_score as ps, recall_score as rs, f1_score as fs
    p = ps(y_val_binary, preds, zero_division=0)
    r = rs(y_val_binary, preds, zero_division=0)
    f = fs(y_val_binary, preds, zero_division=0)
    results.append({"alpha": alpha, "precision": p, "recall": r, "f1": f})
    print(f"  {alpha:>4.1f}  {p:>10.4f}  {r:>8.4f}  {f:>8.4f}")

best = max(results, key=lambda x: x["f1"])
print(f"\n✓ Best alpha on VAL set: {best['alpha']}  (F1={best['f1']:.4f})")
print(f"  → Update config.yaml → fusion.alpha = {best['alpha']}")


In [ ]:
# ── Cell 3.3 — Apply Best Alpha Once on Test Set ────────────────────────────
BEST_ALPHA = best["alpha"]
n_test = min(len(cnn_test_risk), len(lstm_test_scores))
cnn_t  = cnn_test_risk[:n_test]
lstm_t = lstm_test_scores[:n_test]
y_test_binary = (y_test_true[:n_test] != HEALTHY_IDX).astype(int)

fused_test = BEST_ALPHA * cnn_t + (1.0 - BEST_ALPHA) * lstm_t
preds_test = (fused_test >= 0.5).astype(int)

from sklearn.metrics import precision_score as ps, recall_score as rs, f1_score as fs
p = ps(y_test_binary, preds_test, zero_division=0)
r = rs(y_test_binary, preds_test, zero_division=0)
f = fs(y_test_binary, preds_test, zero_division=0)
acc = np.mean(y_test_binary == preds_test)

print("=" * 55)
print(f"  FUSION TEST RESULTS  (alpha={BEST_ALPHA})")
print("=" * 55)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {p:.4f}")
print(f"  Recall    : {r:.4f}")
print(f"  F1        : {f:.4f}")
print("=" * 55)
print(f"\n→ Set config.yaml fusion.alpha = {BEST_ALPHA}")


In [ ]:
# ── Cell 3.4 — Ablation Curve Plot ──────────────────────────────────────────
alphas = [r["alpha"] for r in results]
f1s    = [r["f1"]    for r in results]
precs  = [r["precision"] for r in results]
recs   = [r["recall"]    for r in results]

plt.figure(figsize=(10, 5))
plt.plot(alphas, f1s,    "o-", lw=2, color="#2196F3", label="F1 (val)")
plt.plot(alphas, precs,  "s--", lw=1.5, color="#4CAF50", label="Precision (val)")
plt.plot(alphas, recs,   "^--", lw=1.5, color="#F44336", label="Recall (val)")
plt.axvline(BEST_ALPHA, color="gray", ls="--", lw=1.5, label=f"Best alpha={BEST_ALPHA}")
plt.xlabel("Alpha (CNN weight in fusion)"); plt.ylabel("Score")
plt.title("Fusion Ablation — CNN vs LSTM weight sweep (validation set)")
plt.legend(); plt.grid(alpha=0.3); plt.xticks(alphas)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, "ablation_curve.png"), dpi=150)
plt.show()
print(f"Saved ablation_curve.png")
